# Kaggle Data Pipeline Verification (CPU)

This notebook rigorously tests the upgraded `src/` data pipeline on the Kaggle environment (CPU-only).
It verifies that the dataset layout is correct, the preprocessing multithreading is bound safely, and the PyTorch data loader operates without memory leaks or crashes.

In [ ]:
import os
import sys
import time
import pandas as pd

# 1. Attach Repository Root
REPO_ROOT = '/kaggle/working/Knee_RSNA'
if os.path.exists(REPO_ROOT) and REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
elif os.path.abspath('.') not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

from src import config
from src.preprocess import runner
from src.preprocess.cache import cfg_of
from src.dataset import RSNADataset
from torch.utils.data import DataLoader

print('Pipeline imports successful!')

In [ ]:
# 2. Verify Dataset Location & Layout
KAGGLE_DATA_ROOT = '/kaggle/input/rsna-knee-abnormality-detection'
print(f'Checking dataset root: {KAGGLE_DATA_ROOT}')
if not os.path.exists(KAGGLE_DATA_ROOT):
    print('WARNING: Kaggle dataset root not found. Please attach the RSNA competition dataset.')
else:
    print('Dataset attached. Found the following directories:')
    print(os.listdir(KAGGLE_DATA_ROOT))

In [ ]:
# 3. Test the Preprocessing Indexer
print('\n--- Building/Verifying Series Index ---')
cfg = config.get_cfg('v2')

try:
    # Limiting to 25 studies for a quick test
    index_out = runner.get_index(KAGGLE_DATA_ROOT, '/kaggle/working/idx', splits=('train', 'test'), limit=25, force=True)
    index_df = index_out['ann']
    print(f'\nIndex built successfully! Found {len(index_df)} series across the sampled studies.')
    display(index_df.head())
except Exception as e:
    print(f'Index building failed: {e}')

In [ ]:
# 4. Test Cache Generation & Multithreading safety
print('\n--- Testing Cache Generation (CPU/RAM stress) ---')
CACHE_DIR = '/kaggle/working/test_cache'
os.makedirs(os.path.dirname(CACHE_DIR), exist_ok=True)

# Select a small subset of studies to build
test_studies = index_df['StudyInstanceUID'].unique()[:5]

try:
    # Set workers=2 to test multiprocessing pipeline safety
    cache, stats = runner.run_cache(index_out, 'train', CACHE_DIR, cfg=cfg, workers=2, studies=test_studies, fresh=True)
    print('\nCache generation stats:', stats)
    cache.flush()
except Exception as e:
    print(f'Cache generation failed: {e}')

In [ ]:
# 5. PyTorch DataLoader Benchmarking on CPU
print('\n--- Testing PyTorch DataLoader ---')
train_csv = os.path.join(KAGGLE_DATA_ROOT, 'train.csv')
if os.path.exists(train_csv):
    df_train = pd.read_csv(train_csv)
    # Filter to only the studies we just cached
    df_train = df_train[df_train['StudyInstanceUID'].isin(test_studies)]
    
    dataset = RSNADataset(df_train, CACHE_DIR, cfg, is_train=True, n_windows_use=4, aug=False)
    loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0) # num_workers=0 for safe CPU test
    
    t0 = time.time()
    for batch_idx, (imgs, slot_mask, win_mask, targets, weights) in enumerate(loader):
        print(f'Batch {batch_idx+1} | Images shape: {imgs.shape} | Type: {imgs.dtype}')
        # Break after one batch - success!
        break
        
    print(f'Data loader test passed in {time.time() - t0:.2f}s')
else:
    print('train.csv not found. Are you running on Kaggle with the dataset attached?')